In [1]:
from pathlib import Path

import tifffile

# Update this path to the directory that holds your TIFF frames
DATA_DIR = Path("/Users/xiaogangyang/data/cms_rsm3d/stitched3")

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR}")

# Collect all .tiff files in the target directory
image_paths = sorted(DATA_DIR.glob("*.tiff"))
if not image_paths:
    raise FileNotFoundError(f"No TIFF files found under {DATA_DIR}")

# Read each TIFF into memory keyed by filename
frames = {path.name: tifffile.imread(path) for path in image_paths}

print(f"Loaded {len(frames)} TIFF files from {DATA_DIR}")
for name, array in list(frames.items())[:5]:
    print(f"{name}: shape={array.shape}, dtype={array.dtype}")


Loaded 962 TIFF files from /Users/xiaogangyang/data/cms_rsm3d/stitched3
sbcc_s2_test_pos1_x-16.600_th0.000_5.00s_796715_saxs_stitched_FFremoved.tiff: shape=(1679, 1475), dtype=int32
sbcc_s2_test_pos1_x-16.600_th0.000_5.00s_796716_saxs_stitched_FFremoved.tiff: shape=(1679, 1475), dtype=int32
sbcc_s2_test_pos1_x-16.600_th0.000_5.00s_796717_saxs_stitched_FFremoved.tiff: shape=(1679, 1475), dtype=int32
sbcc_s2_test_pos1_x-16.600_th0.000_5.00s_796718_saxs_stitched_FFremoved.tiff: shape=(1679, 1475), dtype=int32
sbcc_s2_test_pos1_x-16.600_th0.000_5.00s_796719_saxs_stitched_FFremoved.tiff: shape=(1679, 1475), dtype=int32


In [2]:
# Convert the frames dictionary into lists of names and arrays
image_names = list(frames.keys())
image_stack = [frames[name] for name in image_names]

print(f"Collected {len(image_stack)} images")
print(f"First image: {image_names[0]}, shape={image_stack[0].shape}, dtype={image_stack[0].dtype}")

Collected 962 images
First image: sbcc_s2_test_pos1_x-16.600_th0.000_5.00s_796715_saxs_stitched_FFremoved.tiff, shape=(1679, 1475), dtype=int32


In [3]:
import numpy as np
import pandas as pd

# Assign omega values starting at 0 with 0.5° increments
omega = np.arange(len(image_stack), dtype=float) * 0.5

# Build a DataFrame with the requested columns
frames_df = pd.DataFrame({
    "frame_index": np.arange(len(image_stack)),
    "image_stack": image_stack,
    "omega_deg": omega,
})

frames_df.head()

,frame_index,image_stack,omega_deg
0,0,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",0.0
1,1,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",0.5
2,2,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1.0
3,3,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1.5
4,4,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",2.0


In [14]:
print(f"First frame omega: {frames_df.loc[0, 'omega_deg']} deg")

First frame omega: 0.0 deg


In [4]:
import numpy as np
import xrayutilities as xu

# Instrument geometry parameters
wavelength_A = 0.9184          # Angstroms
pixel_size_m = 0.172e-3        # meters
sample_detector_distance_m = 3.03  # meters
beam_center = (733, 1679 - 553)    # (row, col) in pixels

# Prepare xrayutilities conversion (sample axes defined outer→inner)
sample_axes = ['z-', 'y-', 'x+']
detector_axes = ['z+']
beam_direction = (0, 1, 0)  # +Y

ny, nx = frames_df.loc[0, 'image_stack'].shape
qconv = xu.experiment.QConversion(sample_axes, detector_axes, beam_direction, wl=wavelength_A)
qconv.init_area(
    'z-', 'x+',
    cch1=beam_center[0],
    cch2=beam_center[1],
    Nch1=ny,
    Nch2=nx,
    distance=sample_detector_distance_m,
    pwidth1=pixel_size_m,
    pwidth2=pixel_size_m,
    detrot=0.0,
    tiltazimuth=0.0,
    tilt=0.0,
)


def compute_q_space(frame_row):
    """Return q-components and magnitude grids for a single frame."""
    omega_deg = float(frame_row.omega_deg)
    qx, qy, qz = qconv.area(omega_deg, 0.0, 0.0, 0.0, deg=True)
    q_mag = np.sqrt(qx**2 + qy**2 + qz**2)
    return {
        "frame_index": int(frame_row.frame_index),
        "omega_deg": omega_deg,
        "qx": qx,
        "qy": qy,
        "qz": qz,
        "q_mag": q_mag,
        "intensity": frame_row.image_stack,
    }

# Convert every frame in the stack to Q-space coordinates
q_space_frames = [compute_q_space(row) for row in frames_df.itertuples(index=False)]

example = q_space_frames[0]
print(
    f"Frame {example['frame_index']}: q-grid shape = {example['qx'].shape}, "
    f"intensity shape = {example['intensity'].shape}"
)


: 